# A Unified Explainable Ensemble Machine Learning Framework for Early Prediction of Multiple Chronic Diseases Using Routine Clinical Biomarkers

This notebook trains and explains the same ensemble learning pipeline (Random Forest, XGBoost, and a
stacked ensemble of both) separately on three public clinical datasets (Chronic Kidney Disease, Heart
Disease, and Diabetes), then compares results and feature importance across all three. Every chart,
table and dataset preview is saved to the project's `figures/` folder for use in the research paper.

In [1]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

sys.path.append(os.getcwd())
from common import (
    class_balance_plot, correlation_heatmap, train_models,
    evaluate_models, run_shap, save_model, RESULTS,
    dataset_preview_image, summary_statistics_image, missing_values_plot,
    feature_distributions_plot, boxplots_by_class, all_metrics_bar_chart,
    builtin_feature_importance_plot, shap_dependence_plot, workflow_diagram,
)

BASE = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE, "data")
FIGURES_DIR = os.path.join(BASE, "figures")
MODELS_DIR = os.path.join(BASE, "notebook")
os.makedirs(FIGURES_DIR, exist_ok=True)
print("data:", DATA_DIR)
print("figures:", FIGURES_DIR)

data: C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\data
figures: C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures


## 0. Methodology Overview

In [2]:
workflow_diagram(FIGURES_DIR)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\workflow_diagram.png


## 1. Chronic Kidney Disease

400 patients, 24 clinical and laboratory features. Source: UCI Machine Learning Repository (id=336).

In [3]:
ckd_raw = pd.read_csv(os.path.join(DATA_DIR, "ckd.csv"))
ckd_raw.columns = [c.strip() for c in ckd_raw.columns]
ckd_raw["class"] = ckd_raw["class"].astype(str).str.strip().replace({"ckd\t": "ckd"})
print(ckd_raw.shape)
dataset_preview_image(ckd_raw, "Chronic Kidney Disease", FIGURES_DIR, stage="raw")
ckd_raw.head()

(400, 25)
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_dataset_preview_raw.png


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [4]:
numeric_cols = ["age","bp","sg","al","su","bgr","bu","sc","sod","pot","hemo","pcv","wbcc","rbcc"]
categorical_cols = ["rbc","pc","pcc","ba","htn","dm","cad","appet","pe","ane"]

ckd_pre = ckd_raw.copy()
for c in numeric_cols:
    ckd_pre[c] = pd.to_numeric(ckd_pre[c], errors="coerce")
for c in categorical_cols:
    ckd_pre[c] = ckd_pre[c].astype(str).str.strip().replace({"nan": np.nan})
missing_values_plot(ckd_pre, "Chronic Kidney Disease", FIGURES_DIR)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_missing_values.png


In [5]:
ckd = ckd_pre.copy()
for c in numeric_cols:
    ckd[c] = ckd[c].fillna(ckd[c].median())
for c in categorical_cols:
    ckd[c] = ckd[c].fillna(ckd[c].mode()[0])
    ckd[c] = LabelEncoder().fit_transform(ckd[c])

ckd["target"] = (ckd["class"] == "ckd").astype(int)
ckd = ckd.drop(columns=["class"])
print(ckd.isnull().sum().sum(), "missing values remaining")
dataset_preview_image(ckd, "Chronic Kidney Disease", FIGURES_DIR, stage="cleaned")
summary_statistics_image(ckd, "Chronic Kidney Disease", FIGURES_DIR)
ckd.head()

0 missing values remaining
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_dataset_preview_cleaned.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_summary_statistics.png


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,target
0,48.0,80.0,1.020,1.0,0.0,1,1,0,0,121.0,36.0,1.2,138.0,4.4,15.4,44.0,7800.0,5.2,1,1,0,0,0,0,1
1,7.0,50.0,1.020,4.0,0.0,1,1,0,0,121.0,18.0,0.8,138.0,4.4,11.3,38.0,6000.0,4.8,0,0,0,0,0,0,1
2,62.0,80.0,1.010,2.0,3.0,1,1,0,0,423.0,53.0,1.8,138.0,4.4,9.6,31.0,7500.0,4.8,0,1,0,1,0,1,1
3,48.0,70.0,1.005,4.0,0.0,1,0,1,0,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,1,0,0,1,1,1,1
4,51.0,80.0,1.010,2.0,0.0,1,1,0,0,106.0,26.0,1.4,138.0,4.4,11.6,35.0,7300.0,4.6,0,0,0,0,0,0,1


In [6]:
class_balance_plot(ckd["target"], "Chronic Kidney Disease", FIGURES_DIR, positive_label="CKD")
correlation_heatmap(ckd, "Chronic Kidney Disease", FIGURES_DIR)
feature_distributions_plot(ckd, "Chronic Kidney Disease", FIGURES_DIR)

ckd_top_corr = ckd.drop(columns=["target"]).corrwith(ckd["target"]).abs().sort_values(ascending=False).head(4).index.tolist()
boxplots_by_class(ckd, "target", "Chronic Kidney Disease", FIGURES_DIR, ckd_top_corr, positive_label="CKD")
print("Top correlated features:", ckd_top_corr)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_class_balance.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_correlation_heatmap.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_feature_distributions.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_boxplots_by_class.png
Top correlated features: ['hemo', 'pcv', 'sg', 'htn']


In [7]:
X = ckd.drop(columns=["target"])
y = ckd["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

ckd_models = train_models(X_train, y_train)
ckd_best_name, ckd_best_model = evaluate_models(ckd_models, X_test, y_test, "Chronic Kidney Disease", FIGURES_DIR)
all_metrics_bar_chart("Chronic Kidney Disease", FIGURES_DIR)
print("Best model:", ckd_best_name)

Chronic Kidney Disease / Random Forest: acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
Chronic Kidney Disease / XGBoost: acc=0.988 prec=1.000 rec=0.980 f1=0.990 auc=1.000
Chronic Kidney Disease / Stacked Ensemble: acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_confusion_roc.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_all_metrics_comparison.png
Best model: Random Forest


In [8]:
builtin_feature_importance_plot(ckd_best_model, list(X.columns), "Chronic Kidney Disease", FIGURES_DIR, ckd_best_name)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_builtin_feature_importance.png


In [9]:
ckd_ranking = run_shap(ckd_best_model, X_train, X_test, "Chronic Kidney Disease", FIGURES_DIR,
                        model_kind="tree" if ckd_best_name != "Stacked Ensemble" else "other")
ckd_ranking.head(10)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_shap_summary.png


hemo    0.084180
sg      0.077607
pcv     0.069880
sc      0.069799
rbcc    0.046330
al      0.038872
htn     0.038699
dm      0.032619
bu      0.015947
sod     0.015845
dtype: float64

In [10]:
ckd_top_feature = ckd_ranking.index[0]
shap_dependence_plot(ckd_best_model, X_train, X_test, ckd_top_feature, "Chronic Kidney Disease", FIGURES_DIR,
                      model_kind="tree" if ckd_best_name != "Stacked Ensemble" else "other")

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\chronic_kidney_disease_shap_dependence_hemo.png


In [11]:
save_model(ckd_best_model, "Chronic Kidney Disease", MODELS_DIR)
ckd_feature_order = list(X.columns)  # needed later by the Streamlit app

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\notebook\chronic_kidney_disease_model.joblib


## 2. Heart Disease (Cleveland)

303 patients, 13 clinical features. Source: UCI Machine Learning Repository (id=45). The original target is 0-4 (severity); following standard practice for this dataset, it is binarized to presence (1) vs absence (0) of disease.

In [12]:
heart_raw = pd.read_csv(os.path.join(DATA_DIR, "heart.csv"))
print(heart_raw.shape)
dataset_preview_image(heart_raw, "Heart Disease", FIGURES_DIR, stage="raw")
heart_raw.head()

(303, 14)
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_dataset_preview_raw.png


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [13]:
heart_pre = heart_raw.copy()
for c in heart_pre.columns:
    heart_pre[c] = pd.to_numeric(heart_pre[c], errors="coerce")
missing_values_plot(heart_pre, "Heart Disease", FIGURES_DIR)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_missing_values.png


In [14]:
heart = heart_pre.copy()
heart["target"] = (heart["target"] > 0).astype(int)
for c in heart.columns:
    heart[c] = heart[c].fillna(heart[c].median())
dataset_preview_image(heart, "Heart Disease", FIGURES_DIR, stage="cleaned")
summary_statistics_image(heart, "Heart Disease", FIGURES_DIR)
heart.head()

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_dataset_preview_cleaned.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_summary_statistics.png


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [15]:
class_balance_plot(heart["target"], "Heart Disease", FIGURES_DIR, positive_label="Disease Present")
correlation_heatmap(heart, "Heart Disease", FIGURES_DIR)
feature_distributions_plot(heart, "Heart Disease", FIGURES_DIR)

heart_top_corr = heart.drop(columns=["target"]).corrwith(heart["target"]).abs().sort_values(ascending=False).head(4).index.tolist()
boxplots_by_class(heart, "target", "Heart Disease", FIGURES_DIR, heart_top_corr, positive_label="Disease Present")
print("Top correlated features:", heart_top_corr)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_class_balance.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_correlation_heatmap.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_feature_distributions.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_boxplots_by_class.png
Top correlated features: ['thal', 'ca', 'exang', 'oldpeak']


In [16]:
X = heart.drop(columns=["target"])
y = heart["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

heart_models = train_models(X_train, y_train)
heart_best_name, heart_best_model = evaluate_models(heart_models, X_test, y_test, "Heart Disease", FIGURES_DIR)
all_metrics_bar_chart("Heart Disease", FIGURES_DIR)
print("Best model:", heart_best_name)

Heart Disease / Random Forest: acc=0.918 prec=0.871 rec=0.964 f1=0.915 auc=0.958
Heart Disease / XGBoost: acc=0.869 prec=0.794 rec=0.964 f1=0.871 auc=0.931
Heart Disease / Stacked Ensemble: acc=0.902 prec=0.844 rec=0.964 f1=0.900 auc=0.951
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_confusion_roc.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_all_metrics_comparison.png
Best model: Random Forest


In [17]:
builtin_feature_importance_plot(heart_best_model, list(X.columns), "Heart Disease", FIGURES_DIR, heart_best_name)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_builtin_feature_importance.png


In [18]:
heart_ranking = run_shap(heart_best_model, X_train, X_test, "Heart Disease", FIGURES_DIR,
                        model_kind="tree" if heart_best_name != "Stacked Ensemble" else "other")
heart_ranking.head(10)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_shap_summary.png


thal       0.105847
ca         0.099328
cp         0.085578
oldpeak    0.053931
thalach    0.044414
exang      0.039718
slope      0.036064
sex        0.035895
age        0.029452
chol       0.018770
dtype: float64

In [19]:
heart_top_feature = heart_ranking.index[0]
shap_dependence_plot(heart_best_model, X_train, X_test, heart_top_feature, "Heart Disease", FIGURES_DIR,
                      model_kind="tree" if heart_best_name != "Stacked Ensemble" else "other")

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\heart_disease_shap_dependence_thal.png


In [20]:
save_model(heart_best_model, "Heart Disease", MODELS_DIR)
heart_feature_order = list(X.columns)  # needed later by the Streamlit app

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\notebook\heart_disease_model.joblib


## 3. Diabetes

768 patients, 8 clinical features. Source: Pima Indians Diabetes Database. Five columns use 0 as a missing-value placeholder in the raw data (a well-documented quirk of this dataset); those zeros are treated as missing and imputed with the median before training.

In [21]:
diabetes_raw = pd.read_csv(os.path.join(DATA_DIR, "diabetes.csv"))
print(diabetes_raw.shape)
dataset_preview_image(diabetes_raw, "Diabetes", FIGURES_DIR, stage="raw")
diabetes_raw.head()

(768, 9)
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_dataset_preview_raw.png


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [22]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
diabetes_pre = diabetes_raw.copy()
for c in zero_as_missing:
    diabetes_pre[c] = diabetes_pre[c].replace(0, np.nan)
missing_values_plot(diabetes_pre, "Diabetes", FIGURES_DIR)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_missing_values.png


In [23]:
diabetes = diabetes_pre.copy()
for c in zero_as_missing:
    diabetes[c] = diabetes[c].fillna(diabetes[c].median())
diabetes = diabetes.rename(columns={"Outcome": "target"})
dataset_preview_image(diabetes, "Diabetes", FIGURES_DIR, stage="cleaned")
summary_statistics_image(diabetes, "Diabetes", FIGURES_DIR)
diabetes.head()

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_dataset_preview_cleaned.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_summary_statistics.png


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,target
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1


In [24]:
class_balance_plot(diabetes["target"], "Diabetes", FIGURES_DIR, positive_label="Diabetic")
correlation_heatmap(diabetes, "Diabetes", FIGURES_DIR)
feature_distributions_plot(diabetes, "Diabetes", FIGURES_DIR)

diabetes_top_corr = diabetes.drop(columns=["target"]).corrwith(diabetes["target"]).abs().sort_values(ascending=False).head(4).index.tolist()
boxplots_by_class(diabetes, "target", "Diabetes", FIGURES_DIR, diabetes_top_corr, positive_label="Diabetic")
print("Top correlated features:", diabetes_top_corr)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_class_balance.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_correlation_heatmap.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_feature_distributions.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_boxplots_by_class.png
Top correlated features: ['Glucose', 'BMI', 'Age', 'Pregnancies']


In [25]:
X = diabetes.drop(columns=["target"])
y = diabetes["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

diabetes_models = train_models(X_train, y_train)
diabetes_best_name, diabetes_best_model = evaluate_models(diabetes_models, X_test, y_test, "Diabetes", FIGURES_DIR)
all_metrics_bar_chart("Diabetes", FIGURES_DIR)
print("Best model:", diabetes_best_name)

Diabetes / Random Forest: acc=0.734 prec=0.644 rec=0.537 f1=0.586 auc=0.816
Diabetes / XGBoost: acc=0.753 prec=0.660 rec=0.611 f1=0.635 auc=0.822
Diabetes / Stacked Ensemble: acc=0.753 prec=0.682 rec=0.556 f1=0.612 auc=0.823
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_confusion_roc.png
saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_all_metrics_comparison.png
Best model: XGBoost


In [26]:
builtin_feature_importance_plot(diabetes_best_model, list(X.columns), "Diabetes", FIGURES_DIR, diabetes_best_name)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_builtin_feature_importance.png


In [27]:
diabetes_ranking = run_shap(diabetes_best_model, X_train, X_test, "Diabetes", FIGURES_DIR,
                        model_kind="tree" if diabetes_best_name != "Stacked Ensemble" else "other")
diabetes_ranking.head(10)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_shap_summary.png


Glucose                     1.292223
BMI                         0.739598
DiabetesPedigreeFunction    0.465978
Age                         0.401679
Insulin                     0.315452
Pregnancies                 0.195596
BloodPressure               0.191658
SkinThickness               0.140376
dtype: float32

In [28]:
diabetes_top_feature = diabetes_ranking.index[0]
shap_dependence_plot(diabetes_best_model, X_train, X_test, diabetes_top_feature, "Diabetes", FIGURES_DIR,
                      model_kind="tree" if diabetes_best_name != "Stacked Ensemble" else "other")

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\diabetes_shap_dependence_Glucose.png


In [29]:
save_model(diabetes_best_model, "Diabetes", MODELS_DIR)
diabetes_feature_order = list(X.columns)  # needed later by the Streamlit app

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\notebook\diabetes_model.joblib


## 4. Cross-Disease Comparison

This is the section that turns three separate results into one study: a single results table across all nine model runs, and a look at which top SHAP features recur across diseases versus which are disease-specific.

In [30]:
results_df = pd.DataFrame(RESULTS)
results_df.to_csv(os.path.join(FIGURES_DIR, "..", "paper", "all_results.csv"), index=False)
results_df

,Disease,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Chronic Kidney Disease,Random Forest,1.000,1.000,1.000,1.000,1.000
1,Chronic Kidney Disease,XGBoost,0.988,1.000,0.980,0.990,1.000
2,Chronic Kidney Disease,Stacked Ensemble,1.000,1.000,1.000,1.000,1.000
3,Heart Disease,Random Forest,0.918,0.871,0.964,0.915,0.958
4,Heart Disease,XGBoost,0.869,0.794,0.964,0.871,0.931
5,Heart Disease,Stacked Ensemble,0.902,0.844,0.964,0.900,0.951
6,Diabetes,Random Forest,0.734,0.644,0.537,0.586,0.816
7,Diabetes,XGBoost,0.753,0.660,0.611,0.635,0.822
8,Diabetes,Stacked Ensemble,0.753,0.682,0.556,0.612,0.823


In [31]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
pivot = results_df.pivot(index="Model", columns="Disease", values="F1")
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("F1 score")
ax.set_title("F1 Score by Model and Disease")
ax.legend(title="Disease", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
out = os.path.join(FIGURES_DIR, "cross_disease_f1_comparison.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved", out)

saved C:\Users\ezeed\Documents\Global Talent Research - ML Multi-Disease\figures\cross_disease_f1_comparison.png


In [32]:
top5 = pd.DataFrame({
    "Chronic Kidney Disease": ckd_ranking.head(5).index.tolist(),
    "Heart Disease": heart_ranking.head(5).index.tolist(),
    "Diabetes": diabetes_ranking.head(5).index.tolist(),
})
top5.to_csv(os.path.join(FIGURES_DIR, "..", "paper", "top5_features_per_disease.csv"), index=False)
top5

,Chronic Kidney Disease,Heart Disease,Diabetes
0,hemo,thal,Glucose
1,sg,ca,BMI
2,pcv,cp,DiabetesPedigreeFunction
3,sc,oldpeak,Age
4,rbcc,thalach,Insulin


### Notes for the paper

- `figures/` now holds, per disease: a raw and cleaned dataset preview table, a missing-values chart,
  summary statistics, class balance, correlation heatmap, feature distributions, boxplots by class,
  a full metrics comparison, built-in model feature importance, SHAP summary, and a SHAP dependence
  plot for the single most important feature, plus one methodology diagram and the cross-disease F1
  comparison.
- `paper/all_results.csv` holds every metric for all nine model runs (3 diseases x 3 models).
- `paper/top5_features_per_disease.csv` holds the top 5 SHAP-ranked features per disease.
- The three best-performing models are saved in `notebook/` as `.joblib` files for the Streamlit app.